In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# ── Load ──
df = pd.read_csv("../data/smart_home_simulated_v7.csv")
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values(['deviceId', 'ts'])
df = df.ffill()

for col in ['smoke', 'gas', 'water_flow']:
    df[col] = df[col].clip(lower=0)


anomaly_df = df[df['isAnomaly'] == 1].dropna(subset=['anomalyType'])

features = [
    'temp', 'smoke', 'gas', 'power',
    'motion', 'door', 'water_flow'
]

X = anomaly_df[features]
y = anomaly_df['anomalyType']


le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ── Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# ── Train ──
model_type = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='multi:softmax',
    num_class=len(le.classes_)
)

model_type.fit(X_train, y_train)

# ── Evaluate ──
y_pred = model_type.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))


                precision    recall  f1-score   support

energy_anomaly       1.00      1.00      1.00        87
          fire       1.00      1.00      1.00        34
      gas_leak       1.00      1.00      1.00        78
     intrusion       0.99      0.96      0.97        71
  sensor_fault       0.97      0.98      0.98       132
    water_leak       0.99      0.99      0.99       115

      accuracy                           0.99       517
     macro avg       0.99      0.99      0.99       517
  weighted avg       0.99      0.99      0.99       517



In [4]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(model_type, "../models/xgboost_type_v1.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
joblib.dump(features, "../models/features_type.pkl")
print("Classifier saved!")

Classifier saved!
